In [ ]:
%matplotlib inline
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# 1. Cấu hình đồ họa Matplotlib tránh lỗi font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

# 2. Tải Dataset
# LƯU Ý: Giữ file CSV cùng thư mục với file notebook ipynb của bạn
df = pd.read_csv("smartprix_smartphones_april_2026.csv")

# 3. Tiền xử lý nhanh 
df.columns = df.columns.str.strip().str.lower()
mapping = {
    "brand_name": "brand", "model": "model_name", "memory": "storage",
    "battery_capacity(mah)": "battery_capacity", "fast_charging(w)": "fast_charging",
    "processor_name": "processor"
}
df = df.rename(columns={k:v for k,v in mapping.items() if k in df.columns})

if "brand" not in df.columns and "model_name" in df.columns:
    df["brand"] = df["model_name"].apply(lambda x: str(x).split()[0])

if "screen_size" in df.columns:
    df["screen_size"] = pd.to_numeric(df["screen_size"], errors="coerce")
    df.loc[df["screen_size"] > 20, "screen_size"] = np.nan

if "price" in df.columns:
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Ép kiểu cấu hình
for col in ["storage", "ram", "fast_charging"]:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Filling NaN an toàn chống sập nguồn
for col in ["ram", "storage", "fast_charging", "battery_capacity", "screen_size"]:
    if col in df.columns:
        median_val = df[col].median()
        if pd.isna(median_val): median_val = 0
        df[col] = df[col].fillna(median_val)

# Quy đổi INR sang VND (Giả sử x 278.08 VNĐ)
price_col = "price"
currency_label = "VND"
if price_col in df.columns:
    df["price_vnd"] = df["price"] * 278.08
    price_col = "price_vnd"

def format_currency(val):
    return f"{int(round(float(val))):,.0f} ₫" if not pd.isna(val) else ""

print("Data load thành công! Kích thước dữ liệu:", df.shape)
df.head(3) # Preview dữ liệu trực tiếp trên Jupyter


Data load thành công! Kích thước dữ liệu: (997, 27)


,brand,model_name,price_category,price,spec_score,vfm_score,vfm_label,has_5g,has_nfc,has_ir,processor_brand,processor,num_core,processor_speed,ram,storage,battery_capacity,fast_charging,charging_ratio,charging_speed_type,screen_size,refresh_rate,rear_camera,front_camera,rear_camera_count,os,price_vnd
0,oneplus,OnePlus Nord 6,Premium,38999,81,0.906828,Average Value,True,True,True,qualcomm,Snapdragon 8s Gen4,8.0,3.20,8.0,256.0,9000.0,80.0,112.500000,Standard,6.78,165.0,50.0,32.0,2,Android v16,10844841.92
1,samsung,Samsung Galaxy S25 Ultra,Flagship,110000,89,0.658517,Average Value,True,True,False,qualcomm,Snapdragon 8 Elite for Galaxy,8.0,4.47,12.0,256.0,5000.0,45.0,111.111111,Standard,6.90,120.0,200.0,12.0,4,Android v15,30588800.00
2,vivo,Vivo T5 Pro,Mid-Range,29999,72,1.522389,Value King,True,False,True,qualcomm,Snapdragon 7s Gen4,8.0,NaN,8.0,128.0,9020.0,90.0,100.222222,Standard,6.80,120.0,50.0,50.0,3,Android v16,8342121.92


In [ ]:
print("===== CÂU 1: TOP 10 HÃNG ĐIỆN THOẠI =====")
top_brands = df["brand"].value_counts().head(10)
print(top_brands.to_string())

if price_col in df.columns and not df[price_col].dropna().empty:
    avg_price = df.groupby("brand")[price_col].mean().sort_values(ascending=False).head(10)
    print("\nTop 10 hãng theo giá bán trung bình:\n", avg_price.apply(format_currency).to_string())

    # Biểu đồ Giá Trung bình hãng
    plt.figure(figsize=(10, 6))
    sns.barplot(y=avg_price.index, x=avg_price.values, palette="magma", hue=avg_price.index, legend=False)
    plt.title(f"Top 10 hãng có giá TB cao nhất ({currency_label})")
    plt.ylabel("Hãng")
    plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1e6:.1f} Tr" if x >= 1e6 else f"{x:,.0f}"))
    plt.tight_layout()
    display(plt.gcf())
    plt.close()

    # Biểu đồ phân phối mức giá
    plt.figure(figsize=(10, 6))
    sns.histplot(df[price_col].dropna(), bins=30, kde=False, color="blue")
    plt.title(f"Phân phối giá điện thoại ({currency_label})")
    plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1e6:.1f} Tr" if x >= 1e6 else f"{x:,.0f}"))
    plt.tight_layout()
    display(plt.gcf())
    plt.close()


===== CÂU 1: TOP 10 HÃNG ĐIỆN THOẠI =====
brand
vivo        135
realme      127
samsung     127
oppo         90
xiaomi       72
motorola     58
oneplus      54
iqoo         47
poco         45
infinix      38

Top 10 hãng theo giá bán trung bình:
 brand
vertu       127,359,111 ₫
sony         40,457,859 ₫
huawei       32,770,933 ₫
asus         29,890,819 ₫
apple        29,556,253 ₫
tesla        19,465,322 ₫
redmagic     19,462,819 ₫
google       19,354,880 ₫
red          15,013,539 ₫
samsung      12,905,940 ₫


In [ ]:
print("===== CÂU 2: CẤU HÌNH LƯU TRỮ (RAM / ROM) =====")
ram_mode = df["ram"].mode().iloc[0] if not df["ram"].mode().empty else 'N/A'
storage_mode = df["storage"].mode().iloc[0] if not df["storage"].mode().empty else 'N/A'
print(f"RAM phổ biến nhất: {ram_mode} GB")
print(f"Storage phổ biến nhất: {storage_mode} GB")

combo = df.groupby(["ram", "storage"]).size().sort_values(ascending=False).head(10)
print("\nTop 10 tổ hợp RAM/Storage phổ biến nhất:\n", combo.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Ram Plot
ram_counts = df["ram"].value_counts().head(10)
sns.barplot(ax=axes[0], y=ram_counts.index.astype(str), x=ram_counts.values, palette="crest", hue=ram_counts.index.astype(str))
axes[0].set_title("Dung lượng RAM (GB)")

# Storage Plot
storage_counts = df["storage"].value_counts().head(10)
sns.barplot(ax=axes[1], y=storage_counts.index.astype(str), x=storage_counts.values, palette="flare", hue=storage_counts.index.astype(str))
axes[1].set_title("Dung lượng Storage (GB)")

plt.tight_layout()
display(plt.gcf())
plt.close()


===== CÂU 2: CẤU HÌNH LƯU TRỮ (RAM / ROM) =====
RAM phổ biến nhất: 8.0 GB
Storage phổ biến nhất: 128.0 GB

Top 10 tổ hợp RAM/Storage phổ biến nhất:
 ram   storage
8.0   128.0      221
      256.0      214
12.0  256.0      192
6.0   128.0      125
4.0   128.0       71
      64.0        55
12.0  512.0       44
16.0  512.0       37
      256.0       11
3.0   64.0         5


In [8]:
print("===== CÂU 3: XU HƯỚNG MÀN HÌNH & PIN =====")
print(f"Dung lượng pin trung bình: {df['battery_capacity'].mean():.0f} mAh")
print(f"Kích thước màn hình trung bình: {df['screen_size'].mean():.2f} inches")

trend = df[(df.get("battery_capacity", 0) >= 5000) & (df.get("screen_size", 0) >= 6.5)]
print(f"Tỷ lệ máy có Pin>=5000 & Màn hình>=6.5\": {(len(trend)/len(df))*100:.2f}%")

if not trend.empty:
    ex_cols = [c for c in ['brand','model_name','battery_capacity','screen_size', price_col] if c in trend.columns]
    ex_df = trend[ex_cols].head(5).copy()
    if price_col in ex_df.columns: ex_df[price_col] = ex_df[price_col].apply(format_currency)
    print("\nVí dụ mẫu tiêu biểu:\n", ex_df.to_string(index=False))


===== CÂU 3: XU HƯỚNG MÀN HÌNH & PIN =====
Dung lượng pin trung bình: 5910 mAh
Kích thước màn hình trung bình: 6.72 inches
Tỷ lệ máy có Pin>=5000 & Màn hình>=6.5": 87.26%

Ví dụ mẫu tiêu biểu:
   brand               model_name  battery_capacity  screen_size    price_vnd
oneplus           OnePlus Nord 6            9000.0         6.78 10,844,842 ₫
samsung Samsung Galaxy S25 Ultra            5000.0         6.90 30,588,800 ₫
   vivo              Vivo T5 Pro            9020.0         6.80  8,342,122 ₫
   vivo              Vivo T5x 5G            7200.0         6.76  5,283,242 ₫
   vivo               Vivo T5 5G            7500.0         6.67  6,671,139 ₫


In [ ]:
print("===== CÂU 4: MỨC ĐỘ ẢNH HƯỞNG PHẦN CỨNG LÊN GIÁ BÁN =====")
hardware_cols = [c for c in [price_col, "ram", "storage", "battery_capacity", "screen_size"] if c in df.columns]

if len(hardware_cols) >= 2:
    corr_matrix = df[hardware_cols].corr()
    price_corr = corr_matrix[price_col].drop(price_col).sort_values(ascending=False)
    
    for k, v in price_corr.items(): print(f" - {k}: {v:.2f}")
    print(f"\n=> Yếu tố ảnh hưởng mạnh nhất: {price_corr.abs().idxmax()}")
    
    plt.figure(figsize=(8, 5))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Ma trận hệ số tương quan")
    plt.tight_layout()
    display(plt.gcf())
    plt.close()


===== CÂU 4: MỨC ĐỘ ẢNH HƯỞNG PHẦN CỨNG LÊN GIÁ BÁN =====
 - ram: 0.53
 - storage: 0.42
 - screen_size: 0.18
 - battery_capacity: -0.08

=> Yếu tố ảnh hưởng mạnh nhất: ram


In [ ]:
print("===== CÂU 5: THỊ PHẦN PHÂN KHÚC GIÁ =====")
if price_col in df.columns and not df[price_col].dropna().empty:
    max_price = df[price_col].max()
    bins = [0, 4_000_000, 12_000_000, float('inf')] if max_price > 1_000_000 else [0, 14_000, 43_000, float('inf')]
    
    df["price_segment"] = pd.cut(df[price_col], bins=bins, labels=["Phổ thông", "Tầm trung", "Cao cấp"])
    segment_counts = df["price_segment"].value_counts()
    print(segment_counts.to_string())
    
    plt.figure(figsize=(6, 6))
    plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', colors=sns.color_palette("pastel"))
    plt.title("Tỷ lệ các Phân khúc giá")
    display(plt.gcf())
    plt.close()


===== CÂU 5: THỊ PHẦN PHÂN KHÚC GIÁ =====
price_segment
Tầm trung    582
Cao cấp      244
Phổ thông    171


In [13]:
print("===== CÂU 6: ĐẶC QUYỀN MÁY CAO CẤP (FLAGSHIP) =====")
if price_col in df.columns and not df[price_col].dropna().empty:
    threshold = 12_000_000 if df[price_col].max() > 1_000_000 else 43_000
    flagship = df[df[price_col] > threshold]
    
    if len(flagship) > 0:
        print(f"Số lượng Flagship: {len(flagship)} chiếc")
        print(f"RAM tiêu biểu: {int(flagship['ram'].mode().iloc[0]) if not flagship['ram'].mode().empty else 'N/A'} GB")
        print(f"ROM tiêu biểu: {int(flagship['storage'].mode().iloc[0]) if not flagship['storage'].mode().empty else 'N/A'} GB")
        print(f"Chip trùm: {flagship['processor'].mode().iloc[0] if not flagship['processor'].mode().empty else 'N/A'}")
        print(f"Tỷ lệ máy RAM >= 8GB: {(len(flagship[flagship['ram'] >= 8]) / len(flagship) * 100):.1f}%")
        print(f"Tỷ lệ máy Pin >= 5000 mAh: {(len(flagship[flagship['battery_capacity'] >= 5000]) / len(flagship) * 100):.1f}%")


===== CÂU 6: ĐẶC QUYỀN MÁY CAO CẤP (FLAGSHIP) =====
Số lượng Flagship: 244 chiếc
RAM tiêu biểu: 12 GB
ROM tiêu biểu: 256 GB
Chip trùm: Snapdragon 8 Elite Gen 5
Tỷ lệ máy RAM >= 8GB: 97.5%
Tỷ lệ máy Pin >= 5000 mAh: 72.5%


In [ ]:
print("===== CÂU 7: AI/ML DỰ ĐOÁN PHÂN KHÚC (DECISION TREE) =====")
predict_cols = ["fast_charging", "storage", "ram", "processor"]

if set(predict_cols + [price_col]).issubset(df.columns) and not df[price_col].dropna().empty:
    df_c7 = df[predict_cols + ["price_segment"]].dropna().copy()
    X7, y7 = df_c7[predict_cols].copy(), df_c7["price_segment"]
    
    # 🛡️ Data Leakage Fix
    X7_train, X7_test, y7_train, y7_test = train_test_split(X7, y7, test_size=0.2, random_state=42)
    
    if "processor" in X7.columns:
        train_df_temp = df.loc[X7_train.index]
        processor_mapping = train_df_temp.groupby("processor")[price_col].mean()
        fallback = processor_mapping.median() if not processor_mapping.empty else 0
        
        X7_train["processor"] = X7_train["processor"].map(processor_mapping).fillna(fallback)
        X7_test["processor"] = X7_test["processor"].map(processor_mapping).fillna(fallback)
        
    model = DecisionTreeClassifier(max_depth=4, random_state=42)
    model.fit(X7_train, y7_train)
    
    print(f"Mô hình nhận diện độ chính xác ~ {accuracy_score(y7_test, model.predict(X7_test))*100:.1f}%")
    
    report = classification_report(y7_test, model.predict(X7_test), output_dict=True)
    for label in ["Cao cấp", "Tầm trung", "Phổ thông"]:
        if label in report:
            print(f"➤ {label} - Độ bắt trúng: {report[label]['recall']*100:.1f}% | Đoán đúng: {report[label]['precision']*100:.1f}%")
    
    print("\n[Mức độ Quyết định của Phần cứng]")
    for k, v in sorted(dict(zip(predict_cols, model.feature_importances_)).items(), key=lambda x: x[1], reverse=True):
        print(f"   => {k}: {v*100:.1f}%")
        
    plt.figure(figsize=(14, 9))
    plot_tree(model, feature_names=["Sạc(W)", "ROM(GB)", "MB RAM", "Sức ép Chip"], 
              class_names=[str(c) for c in model.classes_], filled=True, rounded=True, fontsize=10)
    plt.title("Mô phỏng Trí tuệ - Cây Quyết định")
    display(plt.gcf())
    plt.close()


===== CÂU 7: AI/ML DỰ ĐOÁN PHÂN KHÚC (DECISION TREE) =====
Mô hình nhận diện độ chính xác ~ 80.0%
➤ Cao cấp - Độ bắt trúng: 58.1% | Đoán đúng: 80.6%
➤ Tầm trung - Độ bắt trúng: 92.9% | Đoán đúng: 76.5%
➤ Phổ thông - Độ bắt trúng: 68.9% | Đoán đúng: 93.9%

[Mức độ Quyết định của Phần cứng]
   => processor: 85.1%
   => ram: 10.8%
   => fast_charging: 3.7%
   => storage: 0.5%
